In [5]:
import yfinance as yf
import pandas as pd
import numpy as np

data = yf.Ticker("^GSPC")
data.info.keys()

dict_keys(['maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLow', 'fiftyDayAverage', 'twoHundredDayAverage', 'currency', 'tradeable', '52WeekChange', 'quoteType', 'symbol', 'language', 'region', 'typeDisp', 'quoteSourceName', 'triggerable', 'customPriceAlertConfidence', 'marketState', 'regularMarketTime', 'exchange', 'messageBoardId', 'exchangeTimezoneName', 'exchangeTimezoneShortName', 'gmtOffSetMilliseconds', 'market', 'esgPopulated', 'hasPrePostMarketData', 'firstTradeDateMilliseconds', 'regularMarketChange', 'regularMarketDayRange', 'fullExchangeName', 'averageDailyVolume3Month', 'fiftyTwoWeekLowChange', 'fiftyTwoWeekLowChangePercent', 'fiftyTwoWeekRange', 'fiftyTwoWeekHi

In [4]:
# Install dependencies for this notebook
%pip install --quiet yfinance


Note: you may need to restart the kernel to use updated packages.


In [6]:
# Get historical data
hist = data.history(period="max")

# Calculate monthly log return (continuous return) based on first and last close
SP500_data = hist.groupby(pd.Grouper(freq='ME')).agg({
    'Close': ['first', 'last']
}).droplevel(0, axis=1)
SP500_data.columns = ['First_Closing_Price', 'Final_Closing_Price']
# Log return: ln(End / Start)
SP500_data['Monthly_Increase'] = np.log(SP500_data['Final_Closing_Price'] / SP500_data['First_Closing_Price'])

SP500_data

,First_Closing_Price,Final_Closing_Price,Monthly_Increase
Date,,,
1927-12-31 00:00:00-05:00,17.660000,17.660000,0.000000
1928-01-31 00:00:00-05:00,17.760000,17.570000,-0.010756
1928-02-29 00:00:00-05:00,17.530001,17.260000,-0.015522
1928-03-31 00:00:00-05:00,17.299999,19.280001,0.108362
1928-04-30 00:00:00-04:00,18.910000,19.750000,0.043463
...,...,...,...
2025-11-30 00:00:00-05:00,6851.970215,6849.089844,-0.000420
2025-12-31 00:00:00-05:00,6812.629883,6845.500000,0.004813
2026-01-31 00:00:00-05:00,6858.470215,6939.029785,0.011678


In [7]:
# Split date into month and year
SP500_data= SP500_data.reset_index()
SP500_data.columns = ['Date', 'First_Closing_Price', 'Final_Closing_Price', 'Monthly_Increase']
SP500_data['Month'] = SP500_data['Date'].dt.month
SP500_data['Year'] = SP500_data['Date'].dt.year

SP500_data

,Date,First_Closing_Price,Final_Closing_Price,Monthly_Increase,Month,Year
0,1927-12-31 00:00:00-05:00,17.660000,17.660000,0.000000,12,1927
1,1928-01-31 00:00:00-05:00,17.760000,17.570000,-0.010756,1,1928
2,1928-02-29 00:00:00-05:00,17.530001,17.260000,-0.015522,2,1928
3,1928-03-31 00:00:00-05:00,17.299999,19.280001,0.108362,3,1928
4,1928-04-30 00:00:00-04:00,18.910000,19.750000,0.043463,4,1928
...,...,...,...,...,...,...
1175,2025-11-30 00:00:00-05:00,6851.970215,6849.089844,-0.000420,11,2025
1176,2025-12-31 00:00:00-05:00,6812.629883,6845.500000,0.004813,12,2025
1177,2026-01-31 00:00:00-05:00,6858.470215,6939.029785,0.011678,1,2026
1178,2026-02-28 00:00:00-05:00,6976.439941,6878.879883,-0.014083,2,2026


In [8]:
# Volatility: annualized std dev of daily returns within each month
daily_returns = hist['Close'].pct_change()
monthly_vol = (
    daily_returns.groupby(pd.Grouper(freq='ME')).std() * (252 ** 0.5)
).rename('Volatility')

# Align by position (same source grouper, same order)
SP500_data = SP500_data.set_index('Date')
SP500_data['Volatility'] = monthly_vol.values
SP500_data = SP500_data.reset_index()

SP500_data

,Date,First_Closing_Price,Final_Closing_Price,Monthly_Increase,Month,Year,Volatility
0,1927-12-31 00:00:00-05:00,17.660000,17.660000,0.000000,12,1927,NaN
1,1928-01-31 00:00:00-05:00,17.760000,17.570000,-0.010756,1,1928,0.121270
2,1928-02-29 00:00:00-05:00,17.530001,17.260000,-0.015522,2,1928,0.103970
3,1928-03-31 00:00:00-05:00,17.299999,19.280001,0.108362,3,1928,0.112672
4,1928-04-30 00:00:00-04:00,18.910000,19.750000,0.043463,4,1928,0.156696
...,...,...,...,...,...,...,...
1175,2025-11-30 00:00:00-05:00,6851.970215,6849.089844,-0.000420,11,2025,0.154245
1176,2025-12-31 00:00:00-05:00,6812.629883,6845.500000,0.004813,12,2025,0.088234
1177,2026-01-31 00:00:00-05:00,6858.470215,6939.029785,0.011678,1,2026,0.104286
1178,2026-02-28 00:00:00-05:00,6976.439941,6878.879883,-0.014083,2,2026,0.135263


In [20]:
def getSP500_data():
    return SP500_data